# CEG-WM Paper Ablation CUDA Notebook

本 Notebook 面向 Google Colab 的论文级单对象单变量消融。

核心语义：
- 单次 base embed；
- 多次 detect rerun；
- 所有 variant 复用同一份 embed_record；
- 输出自动按 variant 后缀区分；
- 汇总 compare summary / compare table 便于论文制表与审计。

本 Notebook 只负责环境准备、参数定义、调用 scripts/run_paper_ablation_workflow.py 与结果展示。

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import psutil

try:
    from google.colab import drive
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

WORK_DIR = Path('/content') if IN_COLAB else Path.cwd()
TARGET_DIR = WORK_DIR / 'CEG-WM'
WORK_DIR.mkdir(parents=True, exist_ok=True)

def run_checked(command, cwd=None):
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        capture_output=True,
        text=True,
        encoding='utf-8',
        errors='replace',
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f'command failed: {command}')
    return result

print('=' * 80)
print('初始化 Paper Ablation Colab 环境')
print('=' * 80)
print(f'IN_COLAB={IN_COLAB}')
print(f'WORK_DIR={WORK_DIR}')
print(f'Python={sys.version.split()[0]}')
print(f'CPU cores={psutil.cpu_count()}')
print(f'RAM total={psutil.virtual_memory().total / (1024 ** 3):.1f} GB')
print(f'RAM available={psutil.virtual_memory().available / (1024 ** 3):.1f} GB')
gpu_probe = subprocess.run(['nvidia-smi'], capture_output=True, text=True, encoding='utf-8', errors='replace')
if gpu_probe.returncode == 0:
    print(gpu_probe.stdout)
else:
    print('nvidia-smi 不可用，当前可能不是 GPU Runtime。')

## 第 1 步：挂载 Google Drive

建议把结果同步到 Google Drive，便于保留 run_root、compare summary 与 CSV。

In [ ]:
USE_GOOGLE_DRIVE = True
GOOGLE_DRIVE_ROOT = None
GOOGLE_DRIVE_EXPORT_DIR = None

print('=' * 80)
print('挂载 Google Drive')
print('=' * 80)

if IN_COLAB and USE_GOOGLE_DRIVE:
    drive.mount('/content/drive', force_remount=False)
    GOOGLE_DRIVE_ROOT = Path('/content/drive/MyDrive')
    GOOGLE_DRIVE_EXPORT_DIR = GOOGLE_DRIVE_ROOT / 'CEG_WM_Outputs' / 'Paper_Ablation_Cuda'
    GOOGLE_DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    print(f'✓ Google Drive 已挂载：{GOOGLE_DRIVE_ROOT}')
    print(f'✓ 导出目录：{GOOGLE_DRIVE_EXPORT_DIR}')
else:
    print('ℹ 跳过 Google Drive 挂载')

## 可编辑 Cell 1：仓库来源模式

这里定义 Colab 中要使用哪一份真实仓库，而不是默认强制 clone / reset。

支持模式：
- local_existing：直接使用当前工作目录里已有仓库；
- git_clone：若目标目录不存在则 clone，若已存在则直接复用；
- git_clone_refresh：显式 fetch 并 reset --hard 到远端分支；
- drive_path：使用 Google Drive 中已有仓库路径；
- uploaded_zip：使用上传 ZIP 或已解压目录。

In [ ]:
REPO_SOURCE_MODE = 'local_existing'
REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
REPO_BRANCH = 'main'
LOCAL_EXISTING_REPO_PATH = ''
DRIVE_REPO_PATH = ''
UPLOADED_ZIP_PATH = ''
UPLOADED_ZIP_EXTRACT_DIR = str(WORK_DIR / 'uploaded_repo')
TARGET_REPO_DIR = str(WORK_DIR / 'CEG-WM')
CONFIG_PATH = 'configs/ablation/paper_ablation_cuda.yaml'
INSTALL_REQUIREMENTS = False
DOWNLOAD_MODELS = True
HF_TOKEN = os.environ.get('HF_TOKEN', '')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

print({
    'REPO_SOURCE_MODE': REPO_SOURCE_MODE,
    'CONFIG_PATH': CONFIG_PATH,
    'LOCAL_EXISTING_REPO_PATH': LOCAL_EXISTING_REPO_PATH or '<auto>',
    'DRIVE_REPO_PATH': DRIVE_REPO_PATH or '<unset>',
    'UPLOADED_ZIP_PATH': UPLOADED_ZIP_PATH or '<unset>',
    'TARGET_REPO_DIR': TARGET_REPO_DIR,
    'INSTALL_REQUIREMENTS': INSTALL_REQUIREMENTS,
    'DOWNLOAD_MODELS': DOWNLOAD_MODELS,
    'HF_TOKEN_CONFIGURED': bool(HF_TOKEN),
})

## 可编辑 Cell 2：实验定义

这里定义论文级同对象消融的运行标签、输出目录、选中 variant、base embed 复用模式，以及是否打包 ZIP。

In [ ]:
OUTPUT_ROOT = 'outputs/Paper_Ablation_Cuda'
RUN_TAG = 'paper_ablation_geo_off_on_demo'
SELECTED_VARIANTS = [
    'GEO-off',
    'GEO-on',
]
BASE_EMBED_REUSE_MODE = 'fresh_run'
REUSE_BASE_EMBED_RECORD_PATH = ''
ENABLE_CALIBRATION = False
ENABLE_EVALUATE = False
PACKAGE_ZIP = True
COPY_RESULTS_TO_DRIVE = True

print({
    'OUTPUT_ROOT': OUTPUT_ROOT,
    'RUN_TAG': RUN_TAG,
    'SELECTED_VARIANTS': SELECTED_VARIANTS,
    'BASE_EMBED_REUSE_MODE': BASE_EMBED_REUSE_MODE,
    'REUSE_BASE_EMBED_RECORD_PATH': REUSE_BASE_EMBED_RECORD_PATH or '<unset>',
    'ENABLE_CALIBRATION': ENABLE_CALIBRATION,
    'ENABLE_EVALUATE': ENABLE_EVALUATE,
    'PACKAGE_ZIP': PACKAGE_ZIP,
    'COPY_RESULTS_TO_DRIVE': COPY_RESULTS_TO_DRIVE,
})

## 可编辑 Cell 3：严格单变量切换定义

这里明确说明本次 on/off 对照切换的是哪一个 detect-side 开关，并强制检查 `SELECTED_VARIANTS` 与该单变量定义一致。

In [ ]:
ABLATION_COMPARISON_NAME = 'GEO repair off/on'
ABLATION_COMPARISON_CATALOG = {
    'GEO repair off/on': {
        'switch_path': 'detect.geometry.geo_score_repair.enabled',
        'variants': ['GEO-off', 'GEO-on'],
        'reason': '仅切换 GEO repair enabled，保持同一对象、同一 base embed、同一 geometry chain。',
    },
    'LF repair off/on': {
        'switch_path': 'detect.content.lf_exact_repair.enabled',
        'variants': ['LF-repair-off', 'LF-repair-on'],
        'reason': '仅切换 LF exact repair enabled，保持同一对象与上游产物不变。',
    },
}

if ABLATION_COMPARISON_NAME not in ABLATION_COMPARISON_CATALOG:
    raise KeyError(f'未知消融定义：{ABLATION_COMPARISON_NAME}')
comparison_def = ABLATION_COMPARISON_CATALOG[ABLATION_COMPARISON_NAME]
ABLATION_SWITCH_NAME = comparison_def['switch_path']
EXPECTED_VARIANTS = comparison_def['variants']
SINGLE_VARIABLE_REASON = comparison_def['reason']

if sorted(SELECTED_VARIANTS) != sorted(EXPECTED_VARIANTS):
    raise ValueError(
        'SELECTED_VARIANTS 必须与单变量定义严格一致：'
        f'selected={SELECTED_VARIANTS}, expected={EXPECTED_VARIANTS}'
    )

print({
    'ABLATION_COMPARISON_NAME': ABLATION_COMPARISON_NAME,
    'ABLATION_SWITCH_NAME': ABLATION_SWITCH_NAME,
    'EXPECTED_VARIANTS': EXPECTED_VARIANTS,
    'SINGLE_VARIABLE_REASON': SINGLE_VARIABLE_REASON,
})

## 第 2 步：解析仓库来源、安装依赖、验证关键模块导入

这一阶段会：
- 按仓库来源模式定位当前真实仓库；
- 可选执行 requirements 安装；
- 验证 run_embed / run_detect / run_paper_ablation_workflow 可导入。

In [ ]:
import zipfile

def _validate_repo_root(repo_root: Path) -> Path:
    repo_root = repo_root.resolve()
    required_paths = [
        repo_root / 'configs' / 'ablation' / 'paper_ablation_cuda.yaml',
        repo_root / 'scripts' / 'run_paper_ablation_workflow.py',
        repo_root / 'notebook' / 'Paper_Ablation_Cuda.ipynb',
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(f'仓库根目录不完整：{missing_paths}')
    return repo_root

def _resolve_repo_from_mode() -> Path:
    target_dir = Path(TARGET_REPO_DIR).expanduser().resolve()
    if REPO_SOURCE_MODE == 'local_existing':
        candidate_roots = []
        if LOCAL_EXISTING_REPO_PATH.strip():
            candidate_roots.append(Path(LOCAL_EXISTING_REPO_PATH).expanduser())
        candidate_roots.extend([Path.cwd(), target_dir])
        for candidate_root in candidate_roots:
            if candidate_root.exists():
                try:
                    return _validate_repo_root(candidate_root)
                except FileNotFoundError:
                    continue
        raise FileNotFoundError('local_existing 模式未找到可用仓库，请设置 LOCAL_EXISTING_REPO_PATH')

    if REPO_SOURCE_MODE == 'git_clone':
        if not target_dir.exists():
            run_checked(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(target_dir)])
        return _validate_repo_root(target_dir)

    if REPO_SOURCE_MODE == 'git_clone_refresh':
        if target_dir.exists() and (target_dir / '.git').exists():
            run_checked(['git', '-C', str(target_dir), 'fetch', 'origin', REPO_BRANCH, '--depth', '1'])
            run_checked(['git', '-C', str(target_dir), 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'])
            run_checked(['git', '-C', str(target_dir), 'reset', '--hard', f'origin/{REPO_BRANCH}'])
        else:
            if target_dir.exists():
                shutil.rmtree(target_dir)
            run_checked(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(target_dir)])
        return _validate_repo_root(target_dir)

    if REPO_SOURCE_MODE == 'drive_path':
        if not DRIVE_REPO_PATH.strip():
            raise ValueError('drive_path 模式必须提供 DRIVE_REPO_PATH')
        return _validate_repo_root(Path(DRIVE_REPO_PATH).expanduser())

    if REPO_SOURCE_MODE == 'uploaded_zip':
        if not UPLOADED_ZIP_PATH.strip():
            raise ValueError('uploaded_zip 模式必须提供 UPLOADED_ZIP_PATH')
        upload_path = Path(UPLOADED_ZIP_PATH).expanduser().resolve()
        if upload_path.is_dir():
            return _validate_repo_root(upload_path)
        extract_dir = Path(UPLOADED_ZIP_EXTRACT_DIR).expanduser().resolve()
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(upload_path), str(extract_dir))
        nested_repo_roots = [path for path in extract_dir.rglob('paper_ablation_cuda.yaml') if 'configs' in path.parts]
        if not nested_repo_roots:
            raise FileNotFoundError(f'上传 ZIP 中未找到目标配置：{upload_path}')
        return _validate_repo_root(nested_repo_roots[0].parents[2])

    raise ValueError(f'未知 REPO_SOURCE_MODE: {REPO_SOURCE_MODE}')

CEG_WM_ROOT = _resolve_repo_from_mode()
TARGET_DIR = CEG_WM_ROOT

if INSTALL_REQUIREMENTS:
    run_checked([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=TARGET_DIR)

if str(TARGET_DIR) not in sys.path:
    sys.path.insert(0, str(TARGET_DIR))

print('=' * 80)
print('验证关键模块导入')
print('=' * 80)
print(f'REPO_SOURCE_MODE={REPO_SOURCE_MODE}')
print(f'CEG_WM_ROOT={CEG_WM_ROOT}')
from main.cli import run_embed as _run_embed_module
from main.cli import run_detect as _run_detect_module
from scripts import run_paper_ablation_workflow as _ablation_workflow_module
print('✓ main.cli.run_embed 导入成功')
print('✓ main.cli.run_detect 导入成功')
print('✓ scripts.run_paper_ablation_workflow 导入成功')

## 第 3 步：下载模型权重与 InSPyReNet

这一阶段检查并准备：
- SD3 相关 Hugging Face 模型缓存；
- InSPyReNet 权重 models/inspyrenet/ckpt_base.pth。

In [ ]:
from huggingface_hub import snapshot_download
import shutil
import urllib.request

CEG_WM_ROOT = TARGET_DIR
MODEL_CACHE_DIR = CEG_WM_ROOT / 'models' / 'hf_cache' / 'sd35_medium'
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(MODEL_CACHE_DIR)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(MODEL_CACHE_DIR)
INSPYRENET_DIR = CEG_WM_ROOT / 'models' / 'inspyrenet'
INSPYRENET_DIR.mkdir(parents=True, exist_ok=True)
INSPYRENET_WEIGHT_PATH = INSPYRENET_DIR / 'ckpt_base.pth'

print('=' * 80)
print('下载模型权重与 InSPyReNet')
print('=' * 80)
print(f'SD3 本地缓存目录：{MODEL_CACHE_DIR}')

if DOWNLOAD_MODELS:
    snapshot_kwargs = {
        'repo_id': 'stabilityai/stable-diffusion-3.5-medium',
        'local_dir': str(MODEL_CACHE_DIR),
        'local_files_only': False,
    }
    if HF_TOKEN:
        snapshot_kwargs['token'] = HF_TOKEN
    try:
        snapshot_path = snapshot_download(**snapshot_kwargs)
        print(f'✓ SD3.5 Medium 已缓存到：{MODEL_CACHE_DIR}')
        print(f'✓ SD3.5 Medium snapshot_path：{snapshot_path}')
    except Exception as exc:
        print(f'⚠ SD3.5 Medium 下载失败：{type(exc).__name__}: {exc}')
        print('  若模型已在 Colab/HF cache 中，可继续；否则后续运行会失败。')

    repo_inspyrenet_model_path = CEG_WM_ROOT / 'models' / 'inspyrenet' / 'ckpt_base.pth'
    cached_inspyrenet_model_path = WORK_DIR / 'models' / 'inspyrenet' / 'ckpt_base.pth'
    inspyrenet_model_path = repo_inspyrenet_model_path or cached_inspyrenet_model_path

    cached_inspyrenet_model_path.parent.mkdir(parents=True, exist_ok=True)
    if repo_inspyrenet_model_path is not None:
        repo_inspyrenet_model_path.parent.mkdir(parents=True, exist_ok=True)

    inspyrenet_repo_tree_url = 'https://huggingface.co/plemeri/InSPyReNet/tree/main'
    default_inspyrenet_url = 'https://huggingface.co/plemeri/InSPyReNet/resolve/main/ckpt_base.pth'
    inspyrenet_model_url = os.environ.get('INSPYRENET_MODEL_URL', default_inspyrenet_url).strip()
    force_inspyrenet_download = os.environ.get('INSPYRENET_FORCE_DOWNLOAD', '').strip().lower() in {'1', 'true', 'yes'}

    def _is_valid_weight_file(file_path) -> bool:
        if file_path is None:
            return False
        return file_path.exists() and file_path.is_file() and file_path.stat().st_size > 0

    def _sync_weight_to_repo(source_path: Path, target_path) -> None:
        if target_path is None:
            return
        if source_path.resolve() == target_path.resolve():
            return
        shutil.copy2(source_path, target_path)

    print(f'workflow 目标权重路径：{inspyrenet_model_path}')
    print(f'持久缓存路径：{cached_inspyrenet_model_path}')
    print(f'仓库页面（核对用）：{inspyrenet_repo_tree_url}')
    print(f'下载地址（程序使用）：{inspyrenet_model_url}')
    print(f'强制重下：{force_inspyrenet_download}')

    existing_weight_path = None
    candidate_paths = []
    if repo_inspyrenet_model_path is not None:
        candidate_paths.append(repo_inspyrenet_model_path)
    candidate_paths.append(cached_inspyrenet_model_path)

    for candidate_path in candidate_paths:
        if _is_valid_weight_file(candidate_path):
            existing_weight_path = candidate_path
            break

    download_required = force_inspyrenet_download or existing_weight_path is None
    if existing_weight_path is not None and not force_inspyrenet_download:
        existing_size = existing_weight_path.stat().st_size
        print(f'✓ 已发现现有 InSPyReNet 权重：{existing_weight_path}（size={existing_size} bytes）')
        _sync_weight_to_repo(existing_weight_path, repo_inspyrenet_model_path)
        if existing_weight_path != cached_inspyrenet_model_path:
            shutil.copy2(existing_weight_path, cached_inspyrenet_model_path)
        print('✓ 已复用现有权重，不执行重复下载')

    if download_required:
        temp_download_path = cached_inspyrenet_model_path.with_suffix(cached_inspyrenet_model_path.suffix + '.downloading')
        if temp_download_path.exists():
            temp_download_path.unlink()

        print(f'开始下载：{inspyrenet_model_url}')
        urllib.request.urlretrieve(inspyrenet_model_url, str(temp_download_path))

        if not _is_valid_weight_file(temp_download_path):
            temp_download_path.unlink(missing_ok=True)
            raise RuntimeError(f'下载完成但权重文件无效：{temp_download_path}')

        temp_download_path.replace(cached_inspyrenet_model_path)
        _sync_weight_to_repo(cached_inspyrenet_model_path, repo_inspyrenet_model_path)
        print('✓ 下载完成')

    if _is_valid_weight_file(repo_inspyrenet_model_path):
        final_weight_path = repo_inspyrenet_model_path
    else:
        final_weight_path = cached_inspyrenet_model_path

    final_weight_size = final_weight_path.stat().st_size if _is_valid_weight_file(final_weight_path) else 0
    print(f'最终 workflow 使用路径：{final_weight_path}')
    print(f'最终权重大小：{final_weight_size} bytes')
    print('✓ InSPyReNet 准备完成')
else:
    print('ℹ 已跳过模型下载')

print(f'SD3 缓存目录存在：{MODEL_CACHE_DIR.exists()}')
print(f'SD3 缓存目录非空：{any(MODEL_CACHE_DIR.iterdir()) if MODEL_CACHE_DIR.exists() else False}')
print(f'InSPyReNet 权重存在：{INSPYRENET_WEIGHT_PATH.exists()}')

## 第 4 步：工作流配置和数据准备

这一阶段会生成 notebook 专用运行时配置快照，并把 ENABLE_CALIBRATION / ENABLE_EVALUATE 写回 workflow 配置段。

In [ ]:
import yaml

VALID_BASE_EMBED_REUSE_MODES = {'fresh_run', 'resume', 'reuse_existing_record'}
if BASE_EMBED_REUSE_MODE not in VALID_BASE_EMBED_REUSE_MODES:
    raise ValueError(
        f'BASE_EMBED_REUSE_MODE 必须属于 {sorted(VALID_BASE_EMBED_REUSE_MODES)}；'
        f'当前为 {BASE_EMBED_REUSE_MODE}'
    )

RUNTIME_CFG_DIR = CEG_WM_ROOT / 'outputs' / '_notebook_runtime'
RUNTIME_CFG_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_CONFIG_PATH = RUNTIME_CFG_DIR / f'{RUN_TAG}_paper_ablation_cuda.yaml'
RUN_OUTPUT_ROOT = (CEG_WM_ROOT / OUTPUT_ROOT / RUN_TAG).resolve()

base_cfg = yaml.safe_load((CEG_WM_ROOT / CONFIG_PATH).read_text(encoding='utf-8'))
workflow_cfg = base_cfg['paper_ablation_workflow']
workflow_cfg['output_root'] = OUTPUT_ROOT
workflow_cfg['run_tag'] = RUN_TAG
workflow_cfg['detect_rerun']['enable_calibration'] = ENABLE_CALIBRATION
workflow_cfg['detect_rerun']['enable_evaluate'] = ENABLE_EVALUATE

notebook_runtime_cfg = workflow_cfg.setdefault('notebook_runtime', {})
notebook_runtime_cfg['repo_source_mode'] = REPO_SOURCE_MODE
notebook_runtime_cfg['run_tag'] = RUN_TAG
notebook_runtime_cfg['selected_variants'] = list(SELECTED_VARIANTS)
notebook_runtime_cfg['base_embed_reuse_mode'] = BASE_EMBED_REUSE_MODE
notebook_runtime_cfg['reuse_base_embed_record'] = REUSE_BASE_EMBED_RECORD_PATH or None
notebook_runtime_cfg['package_zip'] = bool(PACKAGE_ZIP)
notebook_runtime_cfg['export_to_drive'] = bool(COPY_RESULTS_TO_DRIVE)
notebook_runtime_cfg['output_root'] = OUTPUT_ROOT

canonical_model_id = 'stabilityai/stable-diffusion-3.5-medium'
local_model_dir = MODEL_CACHE_DIR.resolve()
local_model_dir_ready = bool(local_model_dir.exists() and any(local_model_dir.iterdir()))
base_cfg['model_source'] = 'hf_hub'
base_cfg['model_id'] = canonical_model_id

paper_cfg = base_cfg.setdefault('paper_faithfulness', {})
paper_cfg['enabled'] = True
paper_cfg.setdefault('alignment_check', True)
paper_cfg.setdefault('check_pipeline_fingerprint', True)
paper_cfg.setdefault('check_trajectory_digest', True)

mask_cfg = base_cfg.setdefault('mask', {})
mask_cfg['semantic_model_source'] = 'inspyrenet'
mask_cfg['semantic_model_path'] = str((CEG_WM_ROOT / 'models' / 'inspyrenet' / 'ckpt_base.pth').resolve())

embed_cfg = base_cfg.setdefault('embed', {})
preview_cfg = embed_cfg.setdefault('preview_generation', {})
preview_cfg['enabled'] = True
preview_cfg.setdefault('artifact_rel_path', 'preview/preview.png')

attestation_cfg = base_cfg.setdefault('attestation', {})
attestation_cfg['enabled'] = True
attestation_cfg.setdefault('k_master_env_var', 'CEG_WM_K_MASTER')
attestation_cfg.setdefault('k_prompt_env_var', 'CEG_WM_K_PROMPT')
attestation_cfg.setdefault('k_seed_env_var', 'CEG_WM_K_SEED')
attestation_cfg.setdefault('signature_algorithm', 'ed25519')
attestation_cfg.setdefault('signed_bundle_schema', 'gen_attest_bundle_v1')
attestation_cfg.setdefault('signer_certificate_schema', 'gen_attest_signer_cert_v1')
attestation_cfg.setdefault('require_signed_bundle_verification', True)
attestation_cfg.setdefault('threshold', 0.65)
attestation_cfg.setdefault('lf_weight', 0.5)
attestation_cfg.setdefault('hf_weight', 0.3)
attestation_cfg.setdefault('geo_weight', 0.2)
attestation_cfg.setdefault('decision_mode', 'content_primary_geo_rescue')
attestation_cfg.setdefault('rescue_band_delta_low', 0.05)
attestation_cfg.setdefault('geo_rescue_min_score', 0.3)
attestation_cfg['use_trajectory_mix'] = False

RUNTIME_CONFIG_PATH.write_text(yaml.safe_dump(base_cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')

print('=' * 80)
print('工作流配置和数据准备')
print('=' * 80)
print(f'REPO_SOURCE_MODE={REPO_SOURCE_MODE}')
print(f'CONFIG_PATH={CONFIG_PATH}')
print(f'RUNTIME_CONFIG_PATH={RUNTIME_CONFIG_PATH}')
print(f'RUN_OUTPUT_ROOT={RUN_OUTPUT_ROOT}')
print(f'ABLATION_COMPARISON_NAME={ABLATION_COMPARISON_NAME}')
print(f'ABLATION_SWITCH_NAME={ABLATION_SWITCH_NAME}')
print(f'SELECTED_VARIANTS={SELECTED_VARIANTS}')
print(f'BASE_EMBED_REUSE_MODE={BASE_EMBED_REUSE_MODE}')
print(f'REUSE_BASE_EMBED_RECORD_PATH={REUSE_BASE_EMBED_RECORD_PATH or "<unset>"}')
print(f'ENABLE_CALIBRATION={ENABLE_CALIBRATION}')
print(f'ENABLE_EVALUATE={ENABLE_EVALUATE}')
print(f'PACKAGE_ZIP={PACKAGE_ZIP}')
print(f'MODEL_SOURCE={base_cfg.get("model_source")}')
print(f'MODEL_ID={base_cfg.get("model_id")}')
print(f'LOCAL_MODEL_DIR={local_model_dir}')
print(f'LOCAL_MODEL_DIR_READY={local_model_dir_ready}')
print(f'PREVIEW_GENERATION={embed_cfg.get("preview_generation")}')
print(f'ATTESTATION_ENABLED={attestation_cfg.get("enabled")}')
print(f'USE_TRAJECTORY_MIX={attestation_cfg.get("use_trajectory_mix")}')
print(f'SEMANTIC_MODEL_PATH={mask_cfg.get("semantic_model_path")}')


## 第 5 步：准备 attestation 环境变量

paper_ablation_cuda 继承 attestation 正式链要求，因此此步会准备 CEG_WM_K_MASTER / CEG_WM_K_PROMPT / CEG_WM_K_SEED。

In [ ]:
import re
import secrets

ATTESTATION_ENV_SPECS = {
    'CEG_WM_K_MASTER': 64,
    'CEG_WM_K_PROMPT': 32,
    'CEG_WM_K_SEED': 32,
}
USER_ATTESTATION_VALUES = {
    'CEG_WM_K_MASTER': '',
    'CEG_WM_K_PROMPT': '',
    'CEG_WM_K_SEED': '',
}
AUTO_GENERATE_MISSING_ATTESTATION_KEYS = True
ATTESTATION_INFO_PATH = WORK_DIR / 'paper_ablation_attestation_env.json'

def _is_valid_hex_secret(value: str, expected_length: int) -> bool:
    if not isinstance(value, str):
        return False
    candidate = value.strip().lower()
    return len(candidate) == expected_length and re.fullmatch(r'[0-9a-f]+', candidate) is not None

def _mask_secret(value: str) -> str:
    return '*' * len(value) if len(value) <= 8 else f'{value[:4]}...{value[-4:]}'

resolved_values = {}
generated_env_vars = []
missing_env_vars = []

for env_name, expected_length in ATTESTATION_ENV_SPECS.items():
    manual_value = USER_ATTESTATION_VALUES.get(env_name, '')
    existing_value = os.environ.get(env_name, '')
    if _is_valid_hex_secret(manual_value, expected_length):
        resolved_value = manual_value.strip().lower()
        source = 'manual_input'
    elif _is_valid_hex_secret(existing_value, expected_length):
        resolved_value = existing_value.strip().lower()
        source = 'existing_env'
    elif AUTO_GENERATE_MISSING_ATTESTATION_KEYS:
        resolved_value = secrets.token_hex(expected_length // 2)
        source = 'generated_for_session'
        generated_env_vars.append(env_name)
    else:
        resolved_value = ''
        source = 'missing'

    if resolved_value:
        os.environ[env_name] = resolved_value
        resolved_values[env_name] = {
            'length': len(resolved_value),
            'masked_value': _mask_secret(resolved_value),
            'source': source,
        }
    else:
        missing_env_vars.append(env_name)

for env_name in ATTESTATION_ENV_SPECS:
    print(f"{env_name}: {resolved_values.get(env_name, {'masked_value': '<absent>', 'source': 'missing'})}")

if missing_env_vars:
    raise RuntimeError(f'attestation 环境变量未就绪：{missing_env_vars}')

ATTESTATION_INFO_PATH.write_text(
    json.dumps({
        'saved_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'generated_env_vars': generated_env_vars,
        'resolved_values': resolved_values,
    }, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(f'✓ attestation 环境信息已保存：{ATTESTATION_INFO_PATH}')

## 第 6 步：运行前预检

在真正执行 ablation workflow 前，检查 GPU、关键脚本、关键模块、配置文件和 InSPyReNet 权重。

In [ ]:
import socket
import yaml

from main.diffusion.sd3 import pipeline_factory

precheck_results = []

def _record_check(name, ok, detail):
    precheck_results.append({'name': name, 'ok': ok, 'detail': detail})
    print(f"{'✓' if ok else '✗'} {name}: {detail}" )

try:
    import torch
    cuda_ok = bool(torch.cuda.is_available())
    _record_check('CUDA 可用', cuda_ok, torch.cuda.get_device_name(0) if cuda_ok else '未检测到 CUDA GPU')
except Exception as exc:
    _record_check('CUDA 可用', False, f'{type(exc).__name__}: {exc}')

required_paths = [
    CEG_WM_ROOT / CONFIG_PATH,
    CEG_WM_ROOT / 'scripts' / 'run_paper_ablation_workflow.py',
    CEG_WM_ROOT / 'main' / 'cli' / 'run_embed.py',
    CEG_WM_ROOT / 'main' / 'cli' / 'run_detect.py',
    CEG_WM_ROOT / 'models' / 'inspyrenet' / 'ckpt_base.pth',
]
for path in required_paths:
    _record_check(f'文件存在: {path.name}', path.exists(), str(path))

required_modules = ['yaml', 'huggingface_hub', 'diffusers', 'transformers', 'safetensors']
for module_name in required_modules:
    try:
        __import__(module_name)
        _record_check(f'模块可导入: {module_name}', True, 'ok')
    except Exception as exc:
        _record_check(f'模块可导入: {module_name}', False, f'{type(exc).__name__}: {exc}')

for env_name, expected_length in ATTESTATION_ENV_SPECS.items():
    value = os.environ.get(env_name, '').strip().lower()
    ok = len(value) == expected_length and all(ch in '0123456789abcdef' for ch in value)
    _record_check(f'attestation 环境变量: {env_name}', ok, f'len={len(value)}' if value else '<absent>')

try:
    from huggingface_hub import HfApi
    socket.setdefaulttimeout(15)
    HfApi().model_info('stabilityai/stable-diffusion-3.5-medium')
    _record_check('HF 模型可访问', True, 'stabilityai/stable-diffusion-3.5-medium')
except Exception as exc:
    _record_check('HF 模型可访问', False, f'{type(exc).__name__}: {exc}')

runtime_cfg = yaml.safe_load(RUNTIME_CONFIG_PATH.read_text(encoding='utf-8'))
runtime_mask_cfg = runtime_cfg.get('mask', {}) if isinstance(runtime_cfg, dict) else {}
runtime_semantic_model_path = runtime_mask_cfg.get('semantic_model_path') if isinstance(runtime_mask_cfg, dict) else None
runtime_semantic_model_ok = isinstance(runtime_semantic_model_path, str) and Path(runtime_semantic_model_path).exists()
_record_check('Runtime semantic_model_path', runtime_semantic_model_ok, str(runtime_semantic_model_path))

runtime_model_source = runtime_cfg.get('model_source') if isinstance(runtime_cfg, dict) else None
runtime_model_id = runtime_cfg.get('model_id') if isinstance(runtime_cfg, dict) else None
supported_runtime_model = runtime_model_source == 'hf_hub' and runtime_model_id == 'stabilityai/stable-diffusion-3.5-medium'
_record_check('Runtime model identity', supported_runtime_model, f'source={runtime_model_source}, model_id={runtime_model_id}')

pipeline_buildable, pipeline_failure_reason, pipeline_failure_summary = pipeline_factory.preflight_pipeline_build(runtime_cfg)
pipeline_detail = 'ok' if pipeline_buildable else f'{pipeline_failure_reason}: {pipeline_failure_summary}'
_record_check('Pipeline 预检', pipeline_buildable, pipeline_detail)

hard_fail = [item for item in precheck_results if not item['ok'] and item['name'] in {
    'CUDA 可用',
    '文件存在: paper_ablation_cuda.yaml',
    '文件存在: run_paper_ablation_workflow.py',
    '文件存在: run_embed.py',
    '文件存在: run_detect.py',
    '文件存在: ckpt_base.pth',
    '模块可导入: diffusers',
    '模块可导入: transformers',
    'attestation 环境变量: CEG_WM_K_MASTER',
    'attestation 环境变量: CEG_WM_K_PROMPT',
    'attestation 环境变量: CEG_WM_K_SEED',
    'Runtime semantic_model_path',
    'Runtime model identity',
    'Pipeline 预检',
}]

if hard_fail:
    raise RuntimeError(f'运行前预检未通过，硬性失败项：{hard_fail}')
else:
    print('✓ 运行前预检通过，可以执行 ablation workflow')


## 第 7 步：运行 paper ablation workflow

脚本会执行：
- base embed 一次；
- 多个 detect variant rerun；
- compare summary / compare table 生成。

In [ ]:
command = [
    sys.executable,
    str(CEG_WM_ROOT / 'scripts' / 'run_paper_ablation_workflow.py'),
    '--config', str(RUNTIME_CONFIG_PATH),
    '--run-root', str(RUN_OUTPUT_ROOT),
]

if BASE_EMBED_REUSE_MODE == 'fresh_run':
    command.append('--fresh-run')
elif BASE_EMBED_REUSE_MODE == 'resume':
    command.append('--resume')
elif BASE_EMBED_REUSE_MODE == 'reuse_existing_record':
    if not REUSE_BASE_EMBED_RECORD_PATH.strip():
        raise ValueError('reuse_existing_record 模式必须提供 REUSE_BASE_EMBED_RECORD_PATH')
    command.extend(['--reuse-base-embed-record', REUSE_BASE_EMBED_RECORD_PATH])
else:
    raise ValueError(f'未知 BASE_EMBED_REUSE_MODE: {BASE_EMBED_REUSE_MODE}')

for variant_name in SELECTED_VARIANTS:
    command.extend(['--variant', variant_name])

print('执行命令：')
print(' '.join(command))
run_checked(command, cwd=CEG_WM_ROOT)
print('✓ Paper ablation workflow 执行完成')

## 第 8 步：结果摘要、关键差异与打包结果

本阶段读取 compare summary / compare table，展示 base embed 与每个 variant 的 detect 结果路径，并输出最终 ZIP 包路径。

In [ ]:
import csv

summary_path = RUN_OUTPUT_ROOT / 'compare' / 'ablation_compare_summary.json'
manifest_path = RUN_OUTPUT_ROOT / 'compare' / 'ablation_manifest.json'
table_path = RUN_OUTPUT_ROOT / 'compare' / 'ablation_compare_table.csv'

summary_obj = json.loads(summary_path.read_text(encoding='utf-8'))
manifest_obj = json.loads(manifest_path.read_text(encoding='utf-8'))
archive_path_value = summary_obj.get('archive_path') or manifest_obj.get('archive_path')
archive_path = Path(archive_path_value) if isinstance(archive_path_value, str) and archive_path_value else None

print('=' * 80)
print('Ablation Summary')
print('=' * 80)
print(f"Base Embed Run Root: {summary_obj['base_embed_run_root']}")
print(f"Base Embed Record: {summary_obj['base_embed_record_path']}")
print(f"Manifest: {manifest_path}")
print(f"Compare Table: {table_path}")
print(f"Archive Path: {archive_path if archive_path is not None else '<not generated>'}")

print('-' * 80)
print('Variant Detect Record Anchors')
for variant_entry in manifest_obj['variants']:
    print(f"  - {variant_entry['suffix']}: {variant_entry['detect_record_path']}")

print('-' * 80)
print('Key GEO/LF Diagnostics')
for variant_row in summary_obj['variants']:
    print(f"Variant: {variant_row['variant_name']}")
    print(f"  Attestation Status: {variant_row.get('attestation_status')}")
    print(f"  Content/Event Score: {variant_row.get('content_attestation_score')} / {variant_row.get('event_attestation_score')}")
    print(
        '  Channel Scores (LF/HF/GEO): '
        f"{variant_row.get('channel_scores_lf')} / {variant_row.get('channel_scores_hf')} / {variant_row.get('channel_scores_geo')}"
    )
    print(f"  Active GEO Score Source: {variant_row.get('active_geo_score_source')}")
    print(
        '  GEO Repair Enabled/Active/Mode: '
        f"{variant_row.get('geo_repair_enabled')} / {variant_row.get('geo_repair_active')} / {variant_row.get('geo_repair_mode')}"
    )
    print(
        '  LF Repair Enabled/Applied/Mode: '
        f"{variant_row.get('lf_exact_repair_enabled')} / {variant_row.get('lf_exact_repair_applied')} / {variant_row.get('lf_exact_repair_mode')}"
    )
    print(f"  Formal Exact Evidence Source: {variant_row.get('formal_exact_evidence_source')}")
    print(f"  Protocol Root Cause: {variant_row.get('protocol_root_cause_classification')}")
    print(f"  Detect Record: {variant_row.get('detect_record_path')}")

print('-' * 80)
print('Compare Table Preview')
with table_path.open('r', encoding='utf-8', newline='') as csv_file:
    table_rows = list(csv.DictReader(csv_file))
for row in table_rows:
    print({
        'variant_name': row.get('variant_name'),
        'attestation_status': row.get('attestation_status'),
        'channel_scores_lf': row.get('channel_scores_lf'),
        'channel_scores_hf': row.get('channel_scores_hf'),
        'channel_scores_geo': row.get('channel_scores_geo'),
        'active_geo_score_source': row.get('active_geo_score_source'),
        'geo_repair_enabled': row.get('geo_repair_enabled'),
        'lf_exact_repair_enabled': row.get('lf_exact_repair_enabled'),
    })

if COPY_RESULTS_TO_DRIVE and GOOGLE_DRIVE_EXPORT_DIR is not None:
    if archive_path is not None and archive_path.exists():
        drive_archive_path = GOOGLE_DRIVE_EXPORT_DIR / archive_path.name
        shutil.copy2(archive_path, drive_archive_path)
        print(f'✓ 压缩包已复制到 Google Drive：{drive_archive_path}')
    drive_summary_path = GOOGLE_DRIVE_EXPORT_DIR / f'{RUN_TAG}_ablation_compare_summary.json'
    drive_manifest_path = GOOGLE_DRIVE_EXPORT_DIR / f'{RUN_TAG}_ablation_manifest.json'
    drive_table_path = GOOGLE_DRIVE_EXPORT_DIR / f'{RUN_TAG}_ablation_compare_table.csv'
    shutil.copy2(summary_path, drive_summary_path)
    shutil.copy2(manifest_path, drive_manifest_path)
    shutil.copy2(table_path, drive_table_path)
    print(f'✓ Compare summary 已复制到 Google Drive：{drive_summary_path}')
    print(f'✓ Manifest 已复制到 Google Drive：{drive_manifest_path}')
    print(f'✓ Compare table 已复制到 Google Drive：{drive_table_path}')

if IN_COLAB and archive_path is not None and archive_path.exists():
    try:
        colab_files.download(str(archive_path))
    except Exception as exc:
        print(f'⚠ 浏览器下载失败：{type(exc).__name__}: {exc}')